# alpha-mipt-rag — прогон на Google Colab (A100, vLLM)

RAG поверх корпуса `alfabank.ru` → submission CSV, на GPU. Код тянется из GitHub.

**Профиль:** embedder `bge-m3` + reranker `bge-reranker-v2-m3` (cuda) + генерация через **vLLM** батчами.
Генератор по умолчанию — `Qwen/Qwen2.5-32B-Instruct-AWQ` (ungated, system-роль OK, влезает в A100 80 ГБ).

**Перед стартом:** Runtime → Change runtime type → **A100 GPU**. Затем Runtime → Run all.
Ячейка зависимостей один раз сама перезапустит runtime — после этого снова Run all.

## 0. GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 1. Параметры

In [ ]:
GIT_URL = "https://github.com/Vesoore/alpha_mipt_rag.git"
WORKDIR = "/content/alpha_mipt_rag"

GEN_MODEL = "Qwen/Qwen2.5-32B-Instruct-AWQ"   # ungated, system-роль OK
HF_TOKEN = ""                                  # нужен только для gated-моделей (gemma и т.п.)

N_QUESTIONS = None   # None = все ~6977; число = быстрая проверка
BATCH = 256          # батч генерации vLLM (на A100 80 ГБ можно 256-512)

print("repo:", GIT_URL, "| model:", GEN_MODEL)

## 2. Клонировать код

In [ ]:
import os, shutil, subprocess, sys

os.chdir("/content")   # не оставаться внутри WORKDIR, иначе rmtree удалит CWD -> git clone exit 128
if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
subprocess.run(["git", "clone", "--depth", "1", GIT_URL, WORKDIR], check=True)

os.chdir(WORKDIR)
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)

for fn in ["websites.csv", "questions.csv", "sample_submission.csv"]:
    p = os.path.join("data", fn)
    assert os.path.exists(p), f"НЕТ {p} — данные должны быть в репозитории"
    print(f"  data/{fn}: {os.path.getsize(p)/1024/1024:.1f} MB")
print("WORKDIR:", os.getcwd())

## 3. Зависимости

vLLM ≥ 0.20 собран под CUDA 13 (`vllm._C` грузит `libcudart.so.13`), а в Colab CUDA 12 → ставим `vllm<0.20` (cu12.8) + cu12.8-torch. Ячейка **один раз** сама перезапустит runtime — после этого снова Runtime → Run all.

In [ ]:
import subprocess, sys, os
MARK = "/content/.deps_done"

# vLLM 0.20+ собирается под CUDA 13 (vllm._C грузит libcudart.so.13), а Colab на CUDA 12.
# Поэтому ставим vllm<0.20 (последние cu12.8-колёса) + cu12.8-torch -> нужен libcudart.so.12, он в Colab есть.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "pip", "install", "--python", sys.executable, "-q",
    "vllm<0.20", "sentence-transformers>=3.0", "faiss-cpu",
    "polars>=1.0", "pydantic>=2.0", "pyyaml", "tqdm", "bm25s>=0.2",
    "--torch-backend=cu128"], check=True)

# torchcodec не нужен для текстового RAG и ломает импорт (нет FFmpeg-библиотек + несовместим с torch 2.10).
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchcodec"], check=False)

import torch
print("torch", torch.__version__, "| cuda", torch.version.cuda)
if not os.path.exists(MARK):
    open(MARK, "w").close()
    print("зависимости установлены — перезапускаю runtime, после рестарта: Runtime → Run all")
    os.kill(os.getpid(), 9)
else:
    import vllm
    print("OK: vllm", vllm.__version__)

In [ ]:
import torch
assert torch.cuda.is_available(), "CUDA недоступна — runtime должен быть A100 GPU"
print(torch.cuda.get_device_name(0))

## 4. HF-токен (только для gated-моделей)

In [ ]:
import os
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("HF login OK")
else:
    print("HF_TOKEN пуст — ок для ungated", GEN_MODEL)

## 5. config.yaml → GPU/vLLM-профиль

Чиним id генератора (в репо стоит несуществующий `gemma-4-27b`) и выставляем `cuda`/`vllm`.

In [ ]:
import yaml
with open("config.yaml", encoding="utf-8") as f:
    c = yaml.safe_load(f)
c["embedder"]["device"] = "cuda"
c["reranker"]["device"] = "cuda"
c["generator"]["backend"] = "vllm"
c["generator"]["model"] = GEN_MODEL
c["generator"]["quantization"] = None
c["generator"]["gpu_memory_utilization"] = 0.85
c["generator"]["max_model_len"] = 8192     # контекстам хватает; режет KV-кэш (дефолт 32768)
c["generator"]["enforce_eager"] = True     # без CUDA-графов — устойчивее старт, меньше памяти
c["generator"]["tensor_parallel_size"] = 1
with open("config.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(c, f, allow_unicode=True, sort_keys=False)
print("generator:", c["generator"]["model"], "| embedder:", c["embedder"]["model"])

## 6. Индекс (chunks + FAISS + BM25)

Артефакты не в git → строим на месте (~3-5 мин).

In [ ]:
from rag.config import load_config, seed_everything
cfg = load_config()
seed_everything(cfg.seed)

needed = [cfg.resolve(cfg.paths.chunks_parquet),
          cfg.resolve(cfg.paths.faiss_index),
          cfg.resolve(cfg.paths.bm25_pickle)]
if [p for p in needed if not p.exists()]:
    print("строю индекс…")
    from rag.data import load_and_clean
    from rag.chunking import chunk_dataframe, save_chunks
    from rag.retrieval.dense import DenseRetriever
    from rag.retrieval.bm25 import BM25Retriever
    docs = load_and_clean(cfg.resolve(cfg.paths.websites_csv), cfg.cleaning)
    chunks_df = chunk_dataframe(docs, cfg.chunking)
    save_chunks(chunks_df, cfg.resolve(cfg.paths.chunks_parquet))
    db = DenseRetriever(cfg.embedder); db.build_index(chunks_df); db.save(cfg.resolve(cfg.paths.faiss_index))
    bb = BM25Retriever(); bb.build_index(chunks_df); bb.save(cfg.resolve(cfg.paths.bm25_pickle))
    del db, bb
    print("индекс готов")
else:
    print("индекс уже на месте")

## 7. Модули пайплайна

Самое долгое — старт vLLM (скачивание весов генератора + прогрев).

In [ ]:
import time
import polars as pl
from transformers import AutoTokenizer
from rag.chunking import load_chunks
from rag.retrieval.dense import DenseRetriever
from rag.retrieval.bm25 import BM25Retriever
from rag.rerank import Reranker
from rag.generation import Generator
from rag.pipeline import Pipeline
from rag.length import trim_answer
from rag.submission import write_submission
from rag.types import Answer

chunks = load_chunks(cfg.resolve(cfg.paths.chunks_parquet))
dense = DenseRetriever(cfg.embedder); dense.load(cfg.resolve(cfg.paths.faiss_index))
bm25 = BM25Retriever(); bm25.load(cfg.resolve(cfg.paths.bm25_pickle))
reranker = Reranker(cfg.reranker)
print(f"retrieval ready: {chunks.height} chunks")

In [ ]:
t0 = time.time()
generator = Generator(cfg.generator, seed=cfg.seed)   # vLLM + скачивание весов
tokenizer = AutoTokenizer.from_pretrained(cfg.chunking.tokenizer_model)
pipeline = Pipeline(cfg, chunks, dense, bm25, reranker, generator, tokenizer)
print(f"generator+pipeline ready in {time.time()-t0:.0f}s")

## 8. Дымовой тест

In [ ]:
ans = pipeline.answer("debug", "Как открыть карту Альфа-Банка?")
print(f"answer ({len(ans.answer)} chars):\n{ans.answer}")

## 9. Полный прогон — батчами + чекпоинт

Генерация батчами через vLLM; ответы пишутся в `submissions/checkpoint.csv` инкрементально. Если сессия не оборвётся — при повторном запуске докатит только недостающие вопросы.

In [ ]:
from pathlib import Path
from tqdm.auto import tqdm

CKPT = Path(WORKDIR) / "submissions" / "checkpoint.csv"
CKPT.parent.mkdir(parents=True, exist_ok=True)
no_data = cfg.length.no_data_phrase

questions = pl.read_csv(cfg.resolve(cfg.paths.questions_csv), infer_schema_length=0)
if N_QUESTIONS is not None:
    questions = questions.head(N_QUESTIONS)

done = set()
if CKPT.exists():
    done = set(pl.read_csv(CKPT, infer_schema_length=0)["q_id"].to_list())
rows = [r for r in questions.iter_rows(named=True) if str(r["q_id"]) not in done]
print(f"всего {questions.height}, сделано {len(done)}, осталось {len(rows)}")

def _append_ckpt(recs):
    df = pl.DataFrame({"q_id": [r[0] for r in recs], "answer": [r[1] for r in recs]})
    if CKPT.exists():
        with open(CKPT, "ab") as f:
            df.write_csv(f, include_header=False)
    else:
        df.write_csv(CKPT)

t0 = time.time()
for i in tqdm(range(0, len(rows), BATCH)):
    batch = rows[i:i + BATCH]
    meta = [(str(r["q_id"]), pipeline.retrieve_and_ground(str(r["q_id"]), r["query"])) for r in batch]
    to_gen = [ctx for _, ctx in meta if ctx is not None]
    gen = iter(generator.generate_batch(to_gen)) if to_gen else iter(())
    recs = []
    for qid, ctx in meta:
        if ctx is None:
            recs.append((qid, no_data))
        else:
            raw = next(gen).strip() or no_data
            recs.append((qid, trim_answer(raw, cfg.length.answer_max_chars)))
    _append_ckpt(recs)

dt = time.time() - t0
print(f"готово: {len(rows)} вопросов за {dt:.0f}s ({dt/max(len(rows),1):.2f}s/q)")

## 10. Submission + статистика

In [ ]:
ck = pl.read_csv(CKPT, infer_schema_length=0)
order = pl.read_csv(cfg.resolve(cfg.paths.questions_csv), infer_schema_length=0).select("q_id")
ck = order.join(ck, on="q_id", how="left")
answers = [Answer(q_id=str(r["q_id"]), answer=(r["answer"] or no_data)) for r in ck.iter_rows(named=True)]

OUT = cfg.resolve(cfg.paths.submission_csv)
write_submission(answers, sample_path=cfg.resolve(cfg.paths.sample_submission_csv), out_path=OUT)

out_df = pl.read_csv(OUT)
tcol = "answer_new" if "answer_new" in out_df.columns else "answer"
lens = out_df.with_columns(pl.col(tcol).str.len_chars().alias("L"))["L"]
nd = out_df.filter(pl.col(tcol) == no_data).height
print(f"rows={out_df.height}  chars: mean={lens.mean():.0f} p50={lens.quantile(0.5):.0f} p95={lens.quantile(0.95):.0f} max={lens.max()}")
print(f"no-data: {nd}/{out_df.height} ({100*nd/out_df.height:.1f}%)")

In [ ]:
from google.colab import files
files.download(str(OUT))   # скачать submission.csv локально

## Перед отправкой (лимит 3 загрузки/день)

- [ ] Полный `questions.csv` прогнан без падений/пустых ответов.
- [ ] Длины адекватны, хвостов ≥3× нет (статистика выше).
- [ ] no-data ответы разумны (не на очевидных вопросах).
- [ ] CSV совпал с `sample_submission.csv` (гарантирует `write_submission`).
- [ ] Залогированы: `GEN_MODEL`, seed, `config.yaml`.